# TB2 – Machine Learning (1FIS0300)
## Modelo 2: Clustering Jerárquico Aglomerativo (Ward)
**Caso de Estudio:** Ministerio de Salud (MINSA) – Certificado de Nacido Vivo (CNV)  
**Objetivo Metodológico:** Segmentación retrospectiva de perfiles maternos y obstétricos asociados a bajo peso al nacer (< 2 500 g) y prematuridad (< 37 semanas)  
**Muestra Establecida:** 30 000 registros independientes (Semilla 42)  
**Responsable:** jfelix torres


### 1. Marco Metodológico y Especificación Técnica

Este notebook implementa el **Clustering Jerárquico Aglomerativo (Ward)** en su forma directa y matemáticamente pura:
* **Muestra:** 30 000 observaciones seleccionadas aleatoriamente sin reemplazo.
* **Algoritmo:** Un único cálculo de matriz de enlace aglomerativo (`scipy.cluster.hierarchy.linkage` con `method='ward'`).
* **Optimización de Arquitectura:** Eliminación de cálculos redundantes (se suprime el segundo ajuste con Scikit-Learn y la propagación de centroides). La jerarquía original se reutiliza para el dendrograma, la exploración de $k$ y la partición final.
* **Anti-Fuga de Información:** Ninguna variable postparto (`PESO_NACIDO`, `TALLA_NACIDO`, `DUR_EMB_PARTO`) interviene en el agrupamiento. Las etiquetas clínicas se emplean únicamente para caracterizar los grupos resultantes.


In [3]:
import os
import sys
import time
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Control de reproducibilidad y entorno gráfico
SEED = 42
N_MUESTRA = 30_000
K_INICIAL = 4

np.random.seed(SEED)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10
PALETA = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

TIEMPOS = {}
t_inicio_total = time.time()
os.makedirs('resultados', exist_ok=True)
print(f'Entorno configurado. Muestra objetivo: {N_MUESTRA:,} registros | Semilla: {SEED}')


Entorno configurado. Muestra objetivo: 30,000 registros | Semilla: 42


### 2. Carga del Dataset y Verificación de Integridad
Búsqueda de `NACIMIENTOS_CNV_LIMPIO.parquet` mediante rutas relativas y verificación de su hash SHA-256.


In [5]:
RUTAS_DATASET = [
    'data/NACIMIENTOS_CNV_LIMPIO.parquet',
    '../NACIMIENTOS_CNV_LIMPIO.parquet',
    'NACIMIENTOS_CNV_LIMPIO.parquet',
    '/content/drive/MyDrive/NACIMIENTOS_CNV_LIMPIO.parquet'
]

ruta_dataset = next((r for r in RUTAS_DATASET if os.path.exists(r)), None)
if ruta_dataset is None:
    raise FileNotFoundError('No se encontró NACIMIENTOS_CNV_LIMPIO.parquet en las rutas previstas.')

t0 = time.time()
with open(ruta_dataset, 'rb') as f:
    sha256_hash = hashlib.sha256(f.read()).hexdigest()

df_raw = pd.read_parquet(ruta_dataset)
TIEMPOS['t_carga'] = time.time() - t0

print(f'Archivo cargado desde: {ruta_dataset}')
print(f'SHA-256: {sha256_hash}')
print(f'Dimensiones leídas: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas')
print(f'Tiempo de lectura y hash: {TIEMPOS["t_carga"]:.2f} s')


Archivo cargado desde: data/NACIMIENTOS_CNV_LIMPIO.parquet
SHA-256: 24d734ca8cc39303da0c08b46007bcc5719e55208024e6e7725ee14e6f1cb55f
Dimensiones leídas: 5,065,065 filas × 24 columnas
Tiempo de lectura y hash: 0.48 s


### 3. Filtros Clínicos y Extracción Directa de 30 000 Registros
Se aplican filtros de plausibilidad biológica y se extraen directamente 30 000 filas reproducibles sin muestra intermedia de 300k.


In [7]:
t0 = time.time()
# Filtros de plausibilidad clínica
n_crudo = len(df_raw)
df_filtrado = df_raw.dropna(subset=['Bajo_Peso_Nacer', 'Prematuro']).copy()
df_filtrado = df_filtrado[df_filtrado.Edad_Madre.between(10, 55)]
df_filtrado = df_filtrado[df_filtrado.PESO_NACIDO.between(300, 6500)]
df_filtrado = df_filtrado[df_filtrado.DUR_EMB_PARTO.between(22, 43)]

n_elegibles = len(df_filtrado)
if n_elegibles < N_MUESTRA:
    raise ValueError(f'Registros elegibles ({n_elegibles}) insuficientes para la muestra de {N_MUESTRA}.')

# Muestreo aleatorio reproducible directo
df_30k = df_filtrado.sample(n=N_MUESTRA, random_state=SEED).copy()

# Extracción de desenlaces clínicos (exclusivamente para evaluación de perfiles)
y_bp = (df_30k.Bajo_Peso_Nacer == 'SI').astype('int8').values
y_pre = (df_30k.Prematuro == 'SI').astype('int8').values

print(f'Registros crudos: {n_crudo:,} | Filtrados: {n_elegibles:,} (eliminados: {n_crudo - n_elegibles:,})')
print(f'Muestra de trabajo: {len(df_30k):,} registros exactos.')
print(f'Tasa Bajo Peso (muestra): {y_bp.mean():.2%}')
print(f'Tasa Prematuridad (muestra): {y_pre.mean():.2%}')
print('Distribución temporal (Año de Nacimiento):')
print(df_30k.FecNac_Año.value_counts().sort_index().to_dict())


Registros crudos: 5,065,065 | Filtrados: 5,063,335 (eliminados: 1,730)
Muestra de trabajo: 30,000 registros exactos.
Tasa Bajo Peso (muestra): 6.26%
Tasa Prematuridad (muestra): 7.09%
Distribución temporal (Año de Nacimiento):
{2015: 2445, 2016: 2680, 2017: 2861, 2018: 2947, 2019: 2893, 2020: 2771, 2021: 2726, 2022: 2666, 2023: 2497, 2024: 2267, 2025: 2317, 2026: 930}


### 4. Ingeniería de Características (13 Variables de Clustering)
Transformación y codificación de las variables maternas y obstétricas prenatales.


In [9]:
EDU_MAP = {
    'NINGUN NIVEL/ILETRADO': 0, 'INCIAL/PRE-ESCOLAR': 0, 'PRIMARIA INCOMPLETA': 1,
    'PRIMARIA COMPLETA': 2, 'SECUNDARIA INCOMPLETA': 3, 'SECUNDARIA COMPLETA': 4,
    'SUPERIOR NO UNIV. INCOMPLETA': 5, 'SUPERIOR UNIV. INCOMPLETA': 5,
    'SUPERIOR NO UNIV. COMPLETA': 6, 'SUPERIOR UNIV. COMPLETA': 7
}
ORD5_MAP = {'1': 1, '2': 2, '3': 3, '4': 4, '>=5': 5}

def parse_perdidas(v):
    if pd.isna(v) or v == 'NINGUNO': return 0
    if str(v).startswith('11'): return 11
    try: return int(v)
    except: return 0

REGION_MAP = {
    '01': 'SELVA', '02': 'SIERRA', '03': 'SIERRA', '04': 'SIERRA', '05': 'SIERRA', '06': 'SIERRA',
    '07': 'COSTA', '08': 'SIERRA', '09': 'SIERRA', '10': 'SIERRA', '11': 'COSTA', '12': 'SIERRA',
    '13': 'COSTA', '14': 'COSTA', '15': 'COSTA', '16': 'SELVA', '17': 'SELVA', '18': 'COSTA',
    '19': 'SIERRA', '20': 'COSTA', '21': 'SIERRA', '22': 'SELVA', '23': 'COSTA', '24': 'COSTA', '25': 'SELVA'
}
NO_CALIF = {'FAMILIAR', 'PARTERA', 'NADIE(AUTOAYUDA)', 'OTRO'}

X = pd.DataFrame(index=df_30k.index)
X['edad_madre']       = df_30k.Edad_Madre.astype('float32')
X['educacion']        = df_30k['Nivel_Intrucción_Madre'].astype(str).map(EDU_MAP).astype('float32')
X['n_embarazos']      = df_30k.Num_embar_madre.map(ORD5_MAP).astype('float32')
X['hijos_vivos']      = df_30k.Hijos_vivo_madre.map(ORD5_MAP).astype('float32')
X['perdidas_previas'] = df_30k.nacmuer_abort_madre.map(parse_perdidas).clip(upper=3).astype('float32')
X['parto_multiple']   = (df_30k.Tipo_Parto.fillna('UNICO') != 'UNICO').astype('int8')
X['cesarea']          = (df_30k.Condicion_Parto == 'CESAREA').astype('int8')
X['fuera_eess']       = (df_30k.Lugar_Nacido != 'ESTABLECIMIENTO DE SALUD').astype('int8')
X['atencion_no_calif']= df_30k.Atiende_Parto.isin(NO_CALIF).astype('int8')
X['sis']              = (df_30k.Financiador_Parto == 'SIS').astype('int8')
X['extranjera']       = (df_30k.Pais_Madre != 'PERU').astype('int8')

depto = df_30k.IdUbigeoInei.astype(str).str[:2]
region = depto.map(REGION_MAP)
X['sierra']           = (region == 'SIERRA').astype('int8')
X['selva']            = (region == 'SELVA').astype('int8')

# Registro y tratamiento de imputaciones
imputaciones = {}
for col in ['educacion', 'n_embarazos', 'hijos_vivos', 'perdidas_previas']:
    n_nulos = int(X[col].isna().sum())
    mediana = float(X[col].median())
    imputaciones[col] = {'nulos': n_nulos, 'mediana_imputada': mediana}
    X[col] = X[col].fillna(mediana)

print('Imputaciones realizadas:', imputaciones)
print(f'Matriz de características lista: {X.shape[1]} columnas.')


Imputaciones realizadas: {'educacion': {'nulos': 18, 'mediana_imputada': 4.0}, 'n_embarazos': {'nulos': 290, 'mediana_imputada': 2.0}, 'hijos_vivos': {'nulos': 251, 'mediana_imputada': 2.0}, 'perdidas_previas': {'nulos': 0, 'mediana_imputada': 0.0}}
Matriz de características lista: 13 columnas.


### 5. Estandarización sobre los 30 000 Registros
Se ajusta `StandardScaler` sobre la muestra de 30k.


In [11]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype('float32')
TIEMPOS['t_preparacion'] = time.time() - t0

print(f'Matriz escalada: {X_scaled.shape} | Memoria: {X_scaled.nbytes / (1024**2):.2f} MB')
print(f'Tiempo de preparación y escalado: {TIEMPOS["t_preparacion"]:.2f} s')


Matriz escalada: (30000, 13) | Memoria: 1.49 MB
Tiempo de preparación y escalado: 3.17 s


### 6. Matriz de Enlace Única (Ward Linkage)
Cálculo de la matriz aglomerativa mediante el criterio de mínima varianza de Ward. Esta matriz se reutiliza para todos los análisis posteriores sin recálculos.


In [13]:
t0 = time.time()
print('Calculando enlace de Ward sobre las 30,000 observaciones...')
Z_ward = linkage(X_scaled, method='ward', metric='euclidean')
TIEMPOS['t_ward'] = time.time() - t0

print(f'Enlace Ward completado en {TIEMPOS["t_ward"]:.2f} segundos.')
print(f'Dimensiones de Z_ward: {Z_ward.shape}')


Calculando enlace de Ward sobre las 30,000 observaciones...
Enlace Ward completado en 10.95 segundos.
Dimensiones de Z_ward: (29999, 4)


### 7. Dendrograma Truncado y Altura de Corte
Se deriva matemáticamente la altura de corte correspondiente a $k = 4$ y se grafica el dendrograma.


In [15]:
# La altura de corte para k grupos se ubica entre las fusiones Z[-(k-1), 2] y Z[-k, 2]
altura_corte_k4 = float((Z_ward[-(4-1), 2] + Z_ward[-4, 2]) / 2.0)

plt.figure(figsize=(10, 5))
plt.title('Dendrograma de Clustering Jerárquico (Ward) – 30 000 Registros', fontsize=12, fontweight='bold')
dendrogram(
    Z_ward,
    truncate_mode='lastp',
    p=25,
    leaf_rotation=45.,
    leaf_font_size=9.,
    show_contracted=True,
    color_threshold=altura_corte_k4
)
plt.xlabel('Clusters condensados (número de observaciones)')
plt.ylabel('Distancia de disimilitud (Ward)')
plt.axhline(y=altura_corte_k4, color='crimson', linestyle='--', label=f'Corte k=4 (distancia = {altura_corte_k4:.2f})')
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig('resultados/dendrograma_ward.png', dpi=150)
plt.show()
print(f'Altura de corte calculada para k=4: {altura_corte_k4:.4f}')


Altura de corte calculada para k=4: 244.9889


<ipython-input-1-76fc446f7311>:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 8. Comparación de Candidatos $k \in [3, 4, 5, 6]$
Se evalúan las particiones derivadas de `Z_ward`. Para garantizar determinismo y agilidad, el Coeficiente de Silueta se calcula sobre una submuestra fija de 5 000 observaciones, mientras que Davies–Bouldin se evalúa sobre los 30 000 registros completos.


In [17]:
t0 = time.time()
rng_eval = np.random.default_rng(SEED)
idx_sil_5k = rng_eval.choice(N_MUESTRA, size=5_000, replace=False)
X_eval_sil = X_scaled[idx_sil_5k]

resultados_k = []
for k_cand in [3, 4, 5, 6]:
    etiquetas_cand = fcluster(Z_ward, t=k_cand, criterion='maxclust')
    sil = float(silhouette_score(X_eval_sil, etiquetas_cand[idx_sil_5k]))
    db = float(davies_bouldin_score(X_scaled, etiquetas_cand))
    resultados_k.append({
        'k': k_cand,
        'Silhouette_5k': sil,
        'Davies_Bouldin_30k': db
    })

df_metricas_k = pd.DataFrame(resultados_k)
df_metricas_k.to_csv('resultados/metricas_k.csv', index=False)
TIEMPOS['t_eval_k'] = time.time() - t0

print('Comparativa de particiones derivadas del enlace Ward:')
print(df_metricas_k.to_string(index=False))
print(f'Tiempo de evaluación de métricas: {TIEMPOS["t_eval_k"]:.2f} s')


Comparativa de particiones derivadas del enlace Ward:
 k  Silhouette_5k  Davies_Bouldin_30k
 3       0.171668            2.043435
 4       0.173350            1.652473
 5       0.191748            1.438086
 6       0.218919            1.295939
Tiempo de evaluación de métricas: 0.57 s


### 9. Partición Final Seleccionada ($k = 4$)
Se extraen las etiquetas de corte para $k = 4$ a partir de la matriz de enlace Ward original (sin segundo entrenamiento ni centroides).


In [19]:
# fcluster asigna 1-indexed (1, 2, 3, 4). Normalizamos a 0-indexed (0, 1, 2, 3).
etiquetas_finales = fcluster(Z_ward, t=K_INICIAL, criterion='maxclust') - 1

conteo_grupos = pd.Series(etiquetas_finales).value_counts().sort_index()
if len(conteo_grupos) != K_INICIAL:
    raise ValueError(f'La partición no produjo exactamente {K_INICIAL} grupos.')

print('Distribución final de los 30 000 registros por cluster:')
for c, cnt in conteo_grupos.items():
    print(f'  Cluster {c}: {cnt:,} casos ({cnt/N_MUESTRA:.2%})')


Distribución final de los 30 000 registros por cluster:
  Cluster 0: 672 casos (2.24%)
  Cluster 1: 18,669 casos (62.23%)
  Cluster 2: 510 casos (1.70%)
  Cluster 3: 10,149 casos (33.83%)


### 10. Validación Extrínseca: Perfiles Clínicos de Riesgo
Caracterización de los 4 clusters según la concentración de Bajo Peso al Nacer y Prematuridad.


In [21]:
df_analisis = pd.DataFrame({
    'id_origen': df_30k.index,
    'cluster': etiquetas_finales,
    'bajo_peso': y_bp,
    'prematuro': y_pre
})

perfiles_clinicos = df_analisis.groupby('cluster').agg(
    total_casos=('bajo_peso', 'count'),
    tasa_bajo_peso=('bajo_peso', 'mean'),
    tasa_prematuro=('prematuro', 'mean')
).reset_index()

perfiles_clinicos['pct_poblacion'] = (perfiles_clinicos['total_casos'] / N_MUESTRA) * 100
perfiles_clinicos['tasa_bajo_peso_pct'] = perfiles_clinicos['tasa_bajo_peso'] * 100
perfiles_clinicos['tasa_prematuro_pct'] = perfiles_clinicos['tasa_prematuro'] * 100

perfiles_clinicos.to_csv('resultados/perfiles_clusters.csv', index=False)

print('=== TASAS DE EVENTOS CLÍNICOS POR CLUSTER ===')
print(perfiles_clinicos[['cluster', 'total_casos', 'pct_poblacion', 'tasa_bajo_peso_pct', 'tasa_prematuro_pct']].round(2).to_string(index=False))

# Gráfico de barras de riesgo clínico
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
sns.barplot(data=perfiles_clinicos, x='cluster', y='tasa_bajo_peso_pct', ax=ax[0], palette=PALETA)
ax[0].set_title('Tasa de Bajo Peso (< 2500 g) por Cluster', fontweight='bold')
ax[0].set_ylabel('% Bajo Peso')
ax[0].axhline(y_bp.mean() * 100, color='red', linestyle='--', label=f'Promedio muestra ({y_bp.mean():.1%})')
ax[0].legend()

sns.barplot(data=perfiles_clinicos, x='cluster', y='tasa_prematuro_pct', ax=ax[1], palette=PALETA)
ax[1].set_title('Tasa de Prematuridad (< 37 s) por Cluster', fontweight='bold')
ax[1].set_ylabel('% Prematuro')
ax[1].axhline(y_pre.mean() * 100, color='red', linestyle='--', label=f'Promedio muestra ({y_pre.mean():.1%})')
ax[1].legend()

plt.tight_layout()
plt.savefig('resultados/tasas_riesgo_clusters.png', dpi=150)
plt.show()


=== TASAS DE EVENTOS CLÍNICOS POR CLUSTER ===
 cluster  total_casos  pct_poblacion  tasa_bajo_peso_pct  tasa_prematuro_pct
       0          672           2.24                7.74                4.32
       1        18669          62.23                5.13                6.03
       2          510           1.70               62.35               60.00
       3        10149          33.83                5.44                6.56


<ipython-input-1-fd4c954cb701>:25: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=perfiles_clinicos, x='cluster', y='tasa_bajo_peso_pct', ax=ax[0], palette=PALETA)
<ipython-input-1-fd4c954cb701>:25: UserWarning: The palette list has more values (6) than needed (4), which may not be intended.
  sns.barplot(data=perfiles_clinicos, x='cluster', y='tasa_bajo_peso_pct', ax=ax[0], palette=PALETA)
<ipython-input-1-fd4c954cb701>:31: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=perfiles_clinicos, x='cluster', y='tasa_prematuro_pct', ax=ax[1], palette=PALETA)
<ipython-input-1-fd4c954cb701>:31: UserWarning: The palette list has more values (6) than needed (4), which may not be intended.
  s

### 11. Caracterización Multivariable y Heatmap Z-Score
Promedios de variables originales y mapa de calor estandarizado por cluster.


In [23]:
df_feat_analisis = X.copy()
df_feat_analisis['cluster'] = etiquetas_finales
medias_originales = df_feat_analisis.groupby('cluster').mean()
print('=== PROMEDIO DE VARIABLES ORIGINALES POR CLUSTER ===')
print(medias_originales.T.round(2))

# Heatmap de perfiles Z-Score
medias_z = pd.DataFrame(X_scaled, columns=list(X.columns)).groupby(etiquetas_finales).mean()
plt.figure(figsize=(12, 4))
sns.heatmap(medias_z, cmap='coolwarm', annot=True, fmt='.2f', cbar=True, center=0)
plt.title('Perfiles Estandarizados (Z-Score) por Cluster Jerárquico', fontsize=12, fontweight='bold')
plt.xlabel('Características de Agrupamiento')
plt.ylabel('Cluster')
plt.tight_layout()
plt.savefig('resultados/heatmap_perfiles_z.png', dpi=150)
plt.show()


=== PROMEDIO DE VARIABLES ORIGINALES POR CLUSTER ===
cluster                0      1      2      3
edad_madre         26.65  26.27  30.28  30.62
educacion           3.06   4.57   4.26   3.48
n_embarazos         2.78   1.81   2.63   3.46
hijos_vivos         2.61   1.57   2.96   3.01
perdidas_previas    0.21   0.21   0.36   0.54
parto_multiple      0.00   0.00   1.00   0.00
cesarea             0.13   0.43   0.88   0.31
fuera_eess          0.83   0.00   0.00   0.00
atencion_no_calif   0.62   0.00   0.01   0.00
sis                 0.52   0.65   0.68   0.80
extranjera          0.01   0.04   0.03   0.00
sierra              0.34   0.40   0.38   0.23
selva               0.29   0.00   0.10   0.34


<ipython-input-1-666fe9295d1d>:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 12. Persistencia de Artefactos y Manifiesto de Ejecución
Exportación del dataset etiquetado y guardado del manifiesto técnico.


In [25]:
# 1. Guardar etiquetas con identificadores de origen
df_analisis.to_csv('resultados/etiquetas_30k.csv', index=False)

# 2. Medir tiempo total
TIEMPOS['t_total_notebook'] = time.time() - t_inicio_total

# 3. Manifiesto
manifiesto = {
    'proyecto': 'TB2 - Machine Learning (UPC)',
    'modelo': 'Clustering Jerárquico Aglomerativo (Ward)',
    'semilla': SEED,
    'n_muestra': N_MUESTRA,
    'k_final': K_INICIAL,
    'dataset_sha256': sha256_hash,
    'versiones': {
        'python': sys.version.split()[0],
        'scipy': getattr(sys.modules.get('scipy'), '__version__', 'N/A'),
        'sklearn': getattr(sys.modules.get('sklearn'), '__version__', 'N/A'),
        'pandas': pd.__version__,
        'numpy': np.__version__
    },
    'metricas_finales': {
        'silhouette_5k': float(df_metricas_k.loc[df_metricas_k.k == 4, 'Silhouette_5k'].values[0]),
        'davies_bouldin_30k': float(df_metricas_k.loc[df_metricas_k.k == 4, 'Davies_Bouldin_30k'].values[0])
    },
    'tiempos_segundos': {k: round(v, 2) for k, v in TIEMPOS.items()},
    'conteo_clusters': conteo_grupos.to_dict(),
    'imputaciones': imputaciones
}

with open('resultados/manifiesto_ejecucion.json', 'w', encoding='utf-8') as f:
    json.dump(manifiesto, f, indent=2, ensure_ascii=False)

print('Manifiesto generado en resultados/manifiesto_ejecucion.json:')
print(json.dumps(manifiesto, indent=2, ensure_ascii=False))


Manifiesto generado en resultados/manifiesto_ejecucion.json:
{
  "proyecto": "TB2 - Machine Learning (UPC)",
  "modelo": "Clustering Jerárquico Aglomerativo (Ward)",
  "semilla": 42,
  "n_muestra": 30000,
  "k_final": 4,
  "dataset_sha256": "24d734ca8cc39303da0c08b46007bcc5719e55208024e6e7725ee14e6f1cb55f",
  "versiones": {
    "python": "3.12.9",
    "scipy": "1.18.1",
    "sklearn": "1.9.1",
    "pandas": "3.0.6",
    "numpy": "2.5.4"
  },
  "metricas_finales": {
    "silhouette_5k": 0.17335040867328644,
    "davies_bouldin_30k": 1.652473499315569
  },
  "tiempos_segundos": {
    "t_carga": 0.48,
    "t_preparacion": 3.17,
    "t_ward": 10.95,
    "t_eval_k": 0.57,
    "t_total_notebook": 18.14
  },
  "conteo_clusters": {
    "0": 672,
    "1": 18669,
    "2": 510,
    "3": 10149
  },
  "imputaciones": {
    "educacion": {
      "nulos": 18,
      "mediana_imputada": 4.0
    },
    "n_embarazos": {
      "nulos": 290,
      "mediana_imputada": 2.0
    },
    "hijos_vivos": {
      "n

### 13. Conclusiones Metodológicas y Discusión

1. **Rigor y Paridad Algorítmica:** La implementación ejecutó un único enlace Ward sobre los 30 000 registros, reutilizándolo para el dendrograma, la exploración $k \in [3, 6]$ y el corte final ($k=4$), garantizando que ninguna observación sea reasignada mediante clasificadores secundarios.
2. **Diferenciación Clínica Retrospectiva:** Se identificó un subgrupo con un marcado incremento en la concentración de desenlaces críticos (embarazo múltiple con elevadas tasas de bajo peso y prematuridad), así como un perfil caracterizado por partos fuera de establecimientos de salud y atención no calificada.
3. **Reproducibilidad y Persistencia:** Todas las transformaciones se ejecutaron a partir de una semilla fija (`SEED = 42`) y sus resultados cuantitativos, gráficos y etiquetas quedaron respaldados en la carpeta `resultados/`.
